In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import warnings
import sys
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session\

In [ ]:
warnings.filterwarnings('ignore')

# =============================================================================
# DOSYA YOLLARI (DİNAMİK)
# =============================================================================
import os
if os.path.exists('/kaggle/input'):
    BASE_PATH = '/kaggle/input/d/mustafassrc/datasest123/'
else:
    BASE_PATH = './data/'

customers_file = BASE_PATH + 'customers.csv'
history_file = BASE_PATH + 'customer_history.csv'
train_labels_file = BASE_PATH + 'referance_data.csv'
test_labels_file = BASE_PATH + 'referance_data_test.csv'
sample_submission_file = BASE_PATH + 'sample_submission.csv'
output_file = "customers_with_features.csv"\

In [ ]:
# =============================================================================
# BÖLÜM 1: VERİ YÜKLEME VE ÖN İŞLEME
# =============================================================================
print("### BÖLÜM 1: VERİ YÜKLEME VE ÖN İŞLEME BAŞLADI ###\n
")

print("Adım 1.1: Veri setleri yükleniyor...")
try:
    customers_df = pd.read_csv(customers_file)
    history_df = pd.read_csv(history_file)
    print("customers.csv ve customer_history.csv başarıyla yüklendi.")
    print(f"Toplam müşteri sayısı (customers.csv): {len(customers_df)}")
    print(f"Toplam işlem kaydı sayısı (history.csv): {len(history_df)}")
except FileNotFoundError as e:
    print(f"HATA: Gerekli dosyalardan biri bulunamadı: {e}")
    sys.exit()

print("\n
Adım 1.2: 'history_df' ön işleniyor...")
# Boş (NaN) değerleri 0 ile doldurma
history_df['cc_transaction_all_amt'] = history_df['cc_transaction_all_amt'].fillna(0)
history_df['cc_transaction_all_cnt'] = history_df['cc_transaction_all_cnt'].fillna(0)
# 'date' sütununu tarih formatına çevirme
history_df['date'] = pd.to_datetime(history_df['date'])
# Toplam işlem tutarı ve adedi için yeni sütunlar oluşturma
history_df['total_amt'] = history_df['mobile_eft_all_amt'] + history_df['cc_transaction_all_amt']
history_df['total_cnt'] = history_df['mobile_eft_all_cnt'] + history_df['cc_transaction_all_cnt']
print("'history_df' ön işleme tamamlandı.")

# =============================================================================
# BÖLÜM 2: REFERANS TARİHİ OLUŞTURMA VE ÖZELLİK MÜHENDİSLİĞİ
# =============================================================================
print("\n
### BÖLÜM 2: ÖZELLİK MÜHENDİSLİĞİ BAŞLADI ###\n
")

print("Adım 2.1: Her müşteri için referans tarihi (son işlem tarihi) belirleniyor...")
# Her müşteri için son işlem tarihini bul
ref_dates = history_df.groupby('cust_id')['date'].max().reset_index()
ref_dates.columns = ['cust_id', 'ref_date']
# Bulunan referans tarihlerini ana history tablosuna ekle
history_df = pd.merge(history_df, ref_dates, on='cust_id', how='left')
print("Referans tarihleri başarıyla eklendi.")


print("\n
Adım 2.2: Davranışsal özellikler türetiliyor (Bu işlem zaman alabilir)...")
grouped = history_df.groupby('cust_id')
features_list = []
total_customers = history_df['cust_id'].nunique()
processed_customers_count = 0

for cust_id, group in grouped:
    processed_customers_count += 1
    if processed_customers_count % 25000 == 0:
        print(f"  İşlenen müşteri: {processed_customers_count}/{total_customers}")

    # Her grubun (müşterinin) kendi referans tarihini al
    ref_date = group['ref_date'].iloc[0]
    
    agg_features = {
        'cust_id': cust_id,
        'toplam_islem_tutari': group['total_amt'].sum(),
        'toplam_islem_adedi': group['total_cnt'].sum(),
        'aylik_ortalama_tutar': group['total_amt'].mean(),
        'aylik_ortalama_islem_adedi': group['total_cnt'].mean(),
        'harcama_istikrari_std': group['total_amt'].std(),
        'ortalama_aktif_urun_sayisi': group['active_product_category_nbr'].mean(),
        'aktif_oldugu_ay_sayisi': group.shape[0],
        'ilk_islem_tarihi': group['date'].min(),
        'son_islem_tarihi': ref_date, # Zaten ref_date ile aynı
        'musteri_omru_gun': (ref_date - group['date'].min()).days
    }
    
    # Zamana bağlı özellikler (son 3, 6, 12 ay)
    for months in [3, 6, 12]:
        start_date = ref_date - pd.DateOffset(months=months)
        period_df = group[group['date'] >= start_date]
        agg_features[f'son_{months}ay_toplam_tutar'] = period_df['total_amt'].sum()
        agg_features[f'son_{months}ay_toplam_islem_adedi'] = period_df['total_cnt'].sum()
        agg_features[f'son_{months}ay_ortalama_tutar'] = period_df['total_amt'].mean()
        
    features_list.append(agg_features)

features_df = pd.DataFrame(features_list).fillna(0)
print("\n
Özellik mühendisliği tamamlandı.")
print(f"Toplam {len(features_df)} müşteri için özellikler türetildi.")

# =============================================================================
# BÖLÜM 3: VERİLERİ BİRLEŞTİRME VE KAYDETME
# =============================================================================
print("\n
### BÖLÜM 3: VERİLERİ BİRLEŞTİRME VE KAYDETME BAŞLADI ###\n
")

print("Adım 3.1: Orijinal müşteri verileri ile yeni özellikleri birleştiriliyor...")
# 'customers_df' ile 'features_df'yi 'cust_id' üzerinden birleştir
# how='left' sayesinde işlem geçmişi olmayan müşteriler bile listede kalır
final_df = pd.merge(customers_df, features_df, on='cust_id', how='left')
print("Birleştirme işlemi tamamlandı.")

# İşlem geçmişi olmayan müşterilerin özellik sütunlarını 0 ile doldur
feature_cols = features_df.columns.drop('cust_id')
final_df[feature_cols] = final_df[feature_cols].fillna(0)
print("İşlem geçmişi olmayan müşteriler için boş değerler dolduruldu.")


print(f"\n
Adım 3.2: Sonuçlar '{output_file}' dosyasına kaydediliyor...")
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n
### TÜM İŞLEMLER BAŞARIYLA TAMAMLANDI! ###")
print(f"Sonuçlar başarıyla kaydedildi. Dosya Boyutları: {final_df.shape[0]} satır, {final_df.shape[1]} sütun.")\

In [ ]:
#Churn eklenmiş hali !!!!!!!!!
import pandas as pd
import sys
features_file = output_file 
# Churn bilgisini içeren eğitim setinizin adı
train_labels_file = train_labels_file
# Sonuçların kaydedileceği yeni dosyanın adı
output_file = "final_dataset_with_churn.csv"

print("İşlem başlıyor...")

try:
    # 1. Mevcut özellik setini yükle
    print(f"'{features_file}' yükleniyor...")
    main_df = pd.read_csv(features_file)

    # 2. Churn bilgisini içeren eğitim setini yükle
    print(f"'{train_labels_file}' yükleniyor...")
    churn_df = pd.read_csv(train_labels_file)
    # Sadece cust_id ve churn sütunlarını alıyoruz, diğerleri zaten var
    churn_df = churn_df[['cust_id', 'churn']]

except FileNotFoundError as e:
    print(f"\n
HATA: Dosya bulunamadı -> {e}")
    print("Lütfen dosya adlarının doğru olduğundan ve kodla aynı klasörde olduklarından emin olun.")
    sys.exit()


# 3. Churn bilgisini ana veri setine ekle (merge işlemi)
print("\n
'churn' verisi ana tabloya ekleniyor...")
# how='left' kullanarak ana tablodaki (main_df) tüm müşterileri koruyoruz.
# Eğitim setinde olmayan (yani test müşterisi olan) müşterilerin 'churn' değeri NaN (boş) olacaktır. Bu doğru bir yaklaşımdır.
final_df = pd.merge(main_df, churn_df, on='cust_id', how='left')

print("Birleştirme tamamlandı.")


# 4. Sonucu yeni bir CSV dosyasına kaydet
print(f"Sonuçlar '{output_file}' dosyasına kaydediliyor...")
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("\n
### İŞLEM BAŞARIYLA TAMAMLANDI! ###")
print(f"'{output_file}' dosyası oluşturuldu. Bu dosya hem özelliklerinizi hem de 'churn' etiketini içermektedir.")

# Doğrulama adımı
churn_counts = final_df['churn'].value_counts(dropna=False)
print("\n
Son dosyadaki 'churn' dağılımı:")
print(churn_counts)\

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Grafiklerin daha güzel görünmesi için stil ayarı
sns.set(style='whitegrid')
plt.style.use('ggplot')

# Veri setimizin dosya yolu
file_path = "/kaggle/input/final-datasets1/final_dataset_with_churn.csv"

# Veri setini yükleyelim
try:
    df = pd.read_csv(file_path)
    print("Veri seti başarıyla yüklendi!")
    print(f"Veri setinin boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
except FileNotFoundError:
    print(f"HATA: '{file_path}' dosya yolunda dosya bulunamadı. Lütfen kontrol edin.")

# Veri setinin ilk 5 satırını göster
print("### Veri Setinin İlk 5 Satırı ###")
display(df.head(15))

# Sütun isimleri, boş olmayan değer sayıları ve veri tipleri hakkında bilgi al
print("\n
### Veri Seti Bilgileri (info) ###")
df.info()\

Düzenlediğim bu veri setinde eksik veri sadece "work_sector" ve "churn" kısmında vardır. "work_sector" değişkenin boş olmasının sebebi meselek olarak öğrenci ve işsiz olarak tanımlanan kişilerin iş sektörü olmamasıdır. O yuzden doldurma işlemi yapılmamalıdır. Churn kısmının boş olmasının sebebi test verisetinden kaynaklı olmasıdır.

In [ ]:
# Churn dağılımını sayısal olarak göster (dropna=False ile NaN değerleri de say)
churn_distribution = df['churn'].value_counts(dropna=False)
churn_percentage = df['churn'].value_counts(normalize=True, dropna=False) * 100

print("### Churn Değişkeni Dağılımı ###")
print(churn_distribution)
print("\n
### Churn Değişkeni Yüzdesel Dağılımı ###")
print(churn_percentage)


# Churn dağılımını görselleştir
plt.figure(figsize=(8, 6))
sns.countplot(x='churn', data=df)
plt.title('Müşteri Churn Dağılımı', fontsize=16)
plt.xlabel('Churn (1: Churn Etti, 0: Etmedi)', fontsize=12)
plt.ylabel('Müşteri Sayısı', fontsize=12)
plt.xticks([0, 1], ['Sadık Müşteri (0)', 'Churn Eden Müşteri (1)'])
plt.show()\

In [ ]:
# Sadece sayısal sütunları seçip temel istatistiklerini hesapla
pd.set_option('display.float_format', lambda x: '%.2f' % x) # Sayıları daha okunaklı formatla
numeric_stats = df.describe().T # .T ile transpozunu alarak daha okunaklı hale getiriyoruz

print("### Sayısal Özelliklerin Temel İstatistikleri ###")
display(numeric_stats)\

In [ ]:
# Analiz edilecek önemli sayısal sütunlar
features_to_plot = [
    'toplam_islem_tutari', 
    'musteri_omru_gun', 
    'aktif_oldugu_ay_sayisi', 
    'son_3ay_toplam_tutar',
    'son_3ay_toplam_islem_adedi'
]

# Histogramları çizdir
df[features_to_plot].hist(bins=30, figsize=(15, 12), layout=(3, 2))
plt.suptitle('Sayısal Özelliklerin Dağılımı (Histogramlar)', size=20, y=1.02)
plt.tight_layout()
plt.show()\

In [ ]:
# Karşılaştırılacak özellikler
features_to_compare = [
    'son_3ay_toplam_tutar',
    'son_3ay_toplam_islem_adedi',
    'toplam_islem_tutari',
    'aktif_oldugu_ay_sayisi',
    'musteri_omru_gun'
]

# Sadece eğitim verisini kullanarak (churn bilgisi olanları) çizim yapalım
train_data = df.dropna(subset=['churn'])

print("### Churn Durumuna Göre Özellik Karşılaştırması (Kutu Grafikleri) ###\n
")

# Her bir özellik için ayrı ayrı kutu grafiği çizdiren döngü
for feature in features_to_compare:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='churn', y=feature, data=train_data)
    plt.title(f"'{feature}' Dağılımı - Churn Durumuna Göre", fontsize=16)
    plt.xlabel('Churn (0: Sadık, 1: Churn Eden)', fontsize=12)
    plt.ylabel(feature, fontsize=12)
    plt.xticks([0, 1], ['Sadık Müşteri (0)', 'Churn Eden Müşteri (1)'])
    
    # Aykırı değerler grafiğin okunmasını zorlaştırıyorsa y-eksenini limitle
    # Örneğin: plt.ylim(0, train_data[feature].quantile(0.95))
    
    plt.show()\

In [ ]:
# 'object' veri tipine sahip sütunları (genellikle kategorik) seçelim
categorical_cols = df.select_dtypes(include=['object']).columns

# Eğitim verisi üzerinden analiz yapıyoruz
train_data = df.dropna(subset=['churn'])

print(f"Tespit edilen kategorik sütunlar: {list(categorical_cols)}\n
")

# Her bir kategorik sütunun dağılımını inceleyelim
for col in categorical_cols:
    print(f"--- '{col}' Sütununun Değer Dağılımı ---")
    print(train_data[col].value_counts())
    
    # Çok fazla benzersiz değer varsa grafiği çizme (örneğin 50'den fazla ise)
    if train_data[col].nunique() < 50:
        plt.figure(figsize=(12, 6))
        sns.countplot(y=col, data=train_data, order=train_data[col].value_counts().index)
        plt.title(f"'{col}' Sütunu Değer Dağılımı", fontsize=16)
        plt.xlabel("Müşteri Sayısı", fontsize=12)
        plt.ylabel(col, fontsize=12)
        plt.show()
    else:
        print(f"'{col}' sütununda çok fazla benzersiz değer olduğu için grafik çizilmedi.\n
")
    print("-" * 40 + "\n
")\

In [ ]:
# Eğitim verisi üzerinden genel (ortalama) churn oranını hesapla
average_churn_rate = train_data['churn'].mean()

print("### Churn Oranının Kategorilere Göre İncelenmesi ###\n
")

for col in categorical_cols:
    # Çok fazla kategori varsa analizi atla
    if train_data[col].nunique() > 50:
        print(f"'{col}' sütunu çok fazla kategori içerdiği için atlandı.\n
")
        continue

    # Her bir kategori için churn oranını hesapla
    churn_by_category = train_data.groupby(col)['churn'].mean().sort_values(ascending=False)
    
    print(f"--- '{col}' Sütununa Göre Churn Oranları ---")
    print(churn_by_category)
    print("-" * 50)
    
    # Sonuçları görselleştir
    plt.figure(figsize=(12, 7))
    sns.barplot(x=churn_by_category.index, y=churn_by_category.values)
    
    # Ortalama churn oranını gösteren kırmızı bir çizgi ekle
    plt.axhline(y=average_churn_rate, color='r', linestyle='--', label=f'Ortalama Churn Oranı ({average_churn_rate:.2%})')
    
    plt.title(f"'{col}' Kategorilerine Göre Churn Oranı", fontsize=16)
    plt.xlabel(col, fontsize=12)
    plt.ylabel("Churn Oranı", fontsize=12)
    plt.xticks(rotation=45, ha='right') # Eksen etiketlerini okunabilir yap
    plt.legend()
    plt.show()
    print("\n
" + "="*60 + "\n
")\

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Veri setini yeniden yüklemeye gerek yok, mevcut 'df' üzerinden devam ediyoruz.

# Sadece sayısal sütunları seçerek korelasyon matrisini hesapla
# 'cust_id' gibi anlamsız ve 'churn' gibi hedef değişkenimizi hariç tutabiliriz.
numerical_features = df.select_dtypes(include=np.number).drop(columns=['cust_id', 'churn'])
correlation_matrix = numerical_features.corr()

# Isı haritasını çizdirme
plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Sayısal Özellikler Arasındaki Korelasyon Matrisi', fontsize=20)
plt.show()\

In [ ]:
# Eğitim verisini alıyoruz (churn bilgisi NaN olmayanlar)
train_data = df.dropna(subset=['churn'])

# Sayısal sütunları ve churn sütununu seçiyoruz
numerical_and_target = train_data.select_dtypes(include=np.number).drop(columns=['cust_id'])

# Tüm bu sütunlar için korelasyon matrisini hesapla
churn_correlation = numerical_and_target.corr()

# Sadece 'churn' sütununu seç, en güçlüden zayıfa sırala ve kendini (churn-churn korelasyonu) listeden çıkar
churn_feature_correlation = churn_correlation['churn'].sort_values(ascending=False).drop('churn')

print("### Tüm Özelliklerin 'Churn' Değişkeni ile Korelasyonu ###\n
")
print(churn_feature_correlation)

# Sonucu bir bar grafiği ile görselleştirelim
plt.figure(figsize=(12, 8))
churn_feature_correlation.plot(kind='bar')
plt.title("Özelliklerin 'Churn' ile Korelasyonu", fontsize=16)
plt.xlabel("Özellikler", fontsize=12)
plt.ylabel("Korelasyon Katsayısı", fontsize=12)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()\

In [ ]:
cols_to_drop = [
    # Analizde zayıf bulunan kategorik özellikler
    'gender', 
  
    
    # Analizde zayıf bulunan sayısal özellik
    'musteri_omru_gun',
    
    # Veri sızıntısı riski veya gereksiz tarih sütunları
    'ilk_islem_tarihi',
    'son_islem_tarihi',
    
    # Modelde kullanılmayacak tanımlayıcı sütun
    'cust_id' 
]

# Sadece veri setinde mevcut olan sütunları listeden al (hata almamak için)
cols_to_drop_existing = [col for col in cols_to_drop if col in df.columns]

print(f"Veri setinden çıkarılacak {len(cols_to_drop_existing)} sütun:")
print(cols_to_drop_existing)

# Sütunları çıkar
df_model_ready = df.drop(columns=cols_to_drop_existing)

print(f"\n
Sütunlar başarıyla çıkarıldı.")
print(f"Modelleme için hazır veri setinin yeni boyutu: {df_model_ready.shape[0]} satır, {df_model_ready.shape[1]} sütun")
display(df_model_ready.head())\

In [ ]:
# Eğitim seti: 'churn' sütunu boş olmayan (dolu) satırlar
train_df = df_model_ready[df_model_ready['churn'].notna()].copy()

# Test seti: 'churn' sütunu boş olan (NaN) satırlar
test_df = df_model_ready[df_model_ready['churn'].isna()].copy()

# Ayırma işleminin doğruluğunu kontrol edelim
print("### Veri Seti Ayırma Sonuçları ###")
print(f"Eğitim setindeki satır sayısı: {train_df.shape[0]}")
print(f"Test setindeki satır sayısı: {test_df.shape[0]}")

# Test setinden artık churn sütununu çıkarabiliriz, çünkü hepsi NaN
test_df = test_df.drop(columns=['churn'])

# Eğitim setinde hedef değişken (y) ve özellikler (X) olarak ayıralım
X = train_df.drop(columns=['churn'])
y = train_df['churn']

# Hedef değişkenin tipini integer yapalım (LGBM için en iyisi)
y = y.astype(int)

print("\n
Eğitim seti, özellikler (X) ve hedef (y) olarak ayrıldı.")
display(X.head())\

In [ ]:
# Eğitim setindeki (X) kategorik sütunları al
categorical_features = X.select_dtypes(include=['object']).columns
print(f"Sayısal formata dönüştürülecek kategorik özellikler: {list(categorical_features)}\n
")

# pd.get_dummies işlemini uygula
X = pd.get_dummies(X, columns=categorical_features, dummy_na=False)
test_df = pd.get_dummies(test_df, columns=categorical_features, dummy_na=False)

# ÖNEMLİ: Eğitim ve test setlerinin sütunlarını birebir aynı hale getir
# Bu adım, eğitim sırasında görülen bir kategorinin testte olmaması (veya tam tersi) durumunu çözer
X, test_df = X.align(test_df, join='inner', axis=1)

print("Kategorik değişkenler başarıyla sayısallaştırıldı.")
print("Eğitim ve Test setlerinin sütunları hizalandı (align edildi).")
print(f"Modelin kullanacağı son özellik sayısı: {X.shape[1]}")
display(X.head())\

In [ ]:
#MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,MODEL EĞİTİMİ NOKTASI,\

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# --- Gerekli Veriler ---
# Bu değişkenlerin bir önceki kod bloklarından geldiğini varsayıyoruz:
# X: Eğitim için hazırlanmış özellikler DataFrame'i
# y: Eğitim için hazırlanmış hedef değişken (churn etiketleri)
# test_df: Tahmin için hazırlanmış test seti DataFrame'i

print("### MODEL EĞİTİM SÜRECİ BAŞLADI ###\n
")

# =============================================================================
# 1. YARIŞMAYA ÖZEL DEĞERLENDİRME METRİĞİ VE PARAMETRELER
# =============================================================================

def weighted_log_loss(y_true, y_pred, k):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    weights = np.where(y_true == 1, 1, k)
    loss = -np.mean(weights * (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)))
    return loss

k = (y == 0).sum() / (y == 1).sum()
print(f"Hesaplanan K (ağırlık) değeri: {k:.4f}")

params = {
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'n_estimators': 2500, 'learning_rate': 0.01, 'num_leaves': 20, 'max_depth': 5,
    'seed': 42, 'n_jobs': -1, 'verbose': -1, 'colsample_bytree': 0.7,
    'subsample': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'scale_pos_weight': k
}

# =============================================================================
# 2. 5-KATLI STRATIFIED ÇAPRAZ DOĞRULAMA İLE MODEL EĞİTİMİ
# =============================================================================

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_df))

print(f"\n
{N_SPLITS}-Katlı Çapraz Doğrulama Başlatılıyor...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"===== Fold {fold+1}/{N_SPLITS} =====")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric='logloss',
              callbacks=[lgb.early_stopping(100, verbose=False)])

    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    test_preds += model.predict_proba(test_df)[:, 1] / N_SPLITS
    
    fold_score = weighted_log_loss(y_val, val_preds, k)
    print(f"Fold {fold+1} Ağırlıklı Log-Loss Skoru: {fold_score:.6f}")

oof_score = weighted_log_loss(y, oof_preds, k)
print(f"\n
Ortalama OOF Ağırlıklı Log-Loss Skoru: {oof_score:.6f}")

# =============================================================================
# 3. GÖNDERİM DOSYASINI OLUŞTURMA (DÜZELTİLMİŞ KISIM)
# =============================================================================

print("\n
Gönderim dosyası oluşturuluyor...")

# Düzeltme: Müşteri ID'lerini ve sıralamasını doğrudan 'sample_submission.csv' dosyasından alıyoruz.
# Bu, yarışma için en doğru ve güvenli yöntemdir.
try:
    # Bu dosya yolu Kaggle ortamına göredir, gerekirse kendi ortamınıza göre düzenleyin.
    sample_submission_df = pd.read_csv(sample_submission_file)
    test_cust_ids = sample_submission_df['cust_id']

    # Submission DataFrame'ini oluştur
    submission_df = pd.DataFrame({'cust_id': test_cust_ids, 'churn': test_preds})
    
    # Çıktı dosyasını kaydet
    submission_df.to_csv('submission.csv', index=False)
    
    print("\n
### TÜM İŞLEMLER BAŞARIYLA TAMAMLANDI! ###")
    print("Tahminleri içeren 'submission.csv' dosyası oluşturuldu ve Kaggle'ın /kaggle/working/ dizinine kaydedildi.")
    display(submission_df.head())

except FileNotFoundError:
    print("\n
HATA: 'sample_submission.csv' dosyası bulunamadı. Lütfen dosya yolunu kontrol edin.")
    print("Gönderim dosyası oluşturulamadı.")\

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# ADIM 1: GÜVENLİ VERİ SETİNİ YÜKLEME VE HAZIRLAMA (DEĞİŞİKLİK YOK)
# =============================================================================
features_file = output_file 
train_labels_file = train_labels_file
df = pd.read_csv(features_file)
churn_df = pd.read_csv(train_labels_file)
churn_df.columns = churn_df.columns.str.strip()
df = pd.merge(df, churn_df[['cust_id', 'churn']], on='cust_id', how='left')
# ... (log dönüşümü, özellik seçimi ve diğer hazırlık adımları burada...)
cols_to_transform = [col for col in df.columns if '_tutar' in col or '_adedi' in col]
for col in cols_to_transform: df[col] = np.log1p(df[col])
cols_to_drop = ['gender', 'province', 'religion', 'work_sector', 'musteri_omru_gun', 'ilk_islem_tarihi', 'son_islem_tarihi', 'cust_id']
cols_to_drop_existing = [col for col in cols_to_drop if col in df.columns]
df_model_ready = df.drop(columns=cols_to_drop_existing)
train_df = df_model_ready[df_model_ready['churn'].notna()].copy()
test_df = df_model_ready[df_model_ready['churn'].isna()].copy()
test_df = test_df.drop(columns=['churn'])
X = train_df.drop(columns=['churn'])
y = train_df['churn'].astype(int)
categorical_features = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_features, dummy_na=False)
test_df = pd.get_dummies(test_df, columns=categorical_features, dummy_na=False)
X, test_df = X.align(test_df, join='inner', axis=1)
print("Veri, modelleme için tamamen hazır!")
k = (y == 0).sum() / (y == 1).sum()

# =============================================================================
# ADIM 2: İKİ FARKLI MODEL İÇİN PARAMETRELERİ TANIMLA
# =============================================================================
# Model A: Orijinal, "genelleyici" parametreler (Kaggle'da daha iyi çalışan)
params_A = {
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'n_estimators': 2500, 'learning_rate': 0.01, 'num_leaves': 20, 'max_depth': 5,
    'seed': 42, 'n_jobs': -1, 'verbose': -1, 'colsample_bytree': 0.7,
    'subsample': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'scale_pos_weight': k
}
# Model B: Optimize edilmiş, "uzman" parametreler (Lokalde daha iyi çalışan)
params_B = {
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'n_estimators': 2500, 'seed': 1337, 'n_jobs': -1, 'verbose': -1, 'scale_pos_weight': k,
    'learning_rate': 0.014373, 'num_leaves': 34, 'max_depth': 3, 'subsample': 0.877,
    'colsample_bytree': 0.637, 'reg_alpha': 0.0289, 'reg_lambda': 0.3654
}
# =============================================================================
# ADIM 3: MODELLERİ EĞİT VE TAHMİNLERİNİ BİRLEŞTİR
# =============================================================================
def train_and_predict(params, model_name, random_seed):
    print(f"\n
### {model_name} Modeli Eğitiliyor... ###")
    test_preds = np.zeros(len(test_df))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    params['seed'] = random_seed # Her modelin farklı bir rastgelelik görmesi için
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  eval_metric='logloss', callbacks=[lgb.early_stopping(100, verbose=False)])
        test_preds += model.predict_proba(test_df)[:, 1] / 5
    return test_preds

test_preds_A = train_and_predict(params_A, "Model A (Generalist)", 42)
test_preds_B = train_and_predict(params_B, "Model B (Uzman)", 1337)

# İki modelin tahminlerini 50/50 ağırlıkla birleştir
blended_preds = (test_preds_A * 0.5) + (test_preds_B * 0.5)
# =============================================================================
# ADIM 4: FİNAL GÖNDERİM DOSYASINI OLUŞTURMA
# =============================================================================
print("\n
Final gönderim dosyası (blended) oluşturuluyor...")
sample_submission_df = pd.read_csv(sample_submission_file)
submission_df = pd.DataFrame({'cust_id': sample_submission_df['cust_id'], 'churn': blended_preds})
submission_df.to_csv('submission_blended_final.csv', index=False)
print("Tahminleri içeren 'submission_blended_final.csv' dosyası oluşturuldu.")\

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score # <--- AUC hesaplaması için
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# ADIM 1: VERİ SETİNİ YÜKLEME VE HAZIRLAMA
# =============================================================================
# Bu kodun Kaggle/benzeri bir ortamda olduğunu varsayıyoruz.
# Dosya yollarını kendi ortamınıza göre düzenlemeniz gerekebilir.
try:
    features_file = output_file 
    train_labels_file = train_labels_file
    sample_submission_file = sample_submission_file

    df = pd.read_csv(features_file)
    churn_df = pd.read_csv(train_labels_file)
    sample_submission_df = pd.read_csv(sample_submission_file)

    churn_df.columns = churn_df.columns.str.strip()
    df = pd.merge(df, churn_df[['cust_id', 'churn']], on='cust_id', how='left')

    # Özellik mühendisliği ve dönüşümler
    cols_to_transform = [col for col in df.columns if '_tutar' in col or '_adedi' in col]
    for col in cols_to_transform: 
        df[col] = np.log1p(df[col])

    cols_to_drop = ['gender', 'province', 'religion', 'work_sector', 'musteri_omru_gun', 'ilk_islem_tarihi', 'son_islem_tarihi', 'cust_id']
    cols_to_drop_existing = [col for col in cols_to_drop if col in df.columns]
    df_model_ready = df.drop(columns=cols_to_drop_existing)

    train_df = df_model_ready[df_model_ready['churn'].notna()].copy()
    test_df = df_model_ready[df_model_ready['churn'].isna()].copy()
    test_df = test_df.drop(columns=['churn'])
    
    X = train_df.drop(columns=['churn'])
    y = train_df['churn'].astype(int)

    # Kategorik özellikler için One-Hot Encoding
    categorical_features = X.select_dtypes(include=['object']).columns
    X = pd.get_dummies(X, columns=categorical_features, dummy_na=False)
    test_df = pd.get_dummies(test_df, columns=categorical_features, dummy_na=False)
    
    # Sütunları hizala
    X, test_df = X.align(test_df, join='inner', axis=1, fill_value=0)

    print("Veri, modelleme için tamamen hazır!")
    print(f"Eğitim seti boyutu (X): {X.shape}")
    print(f"Test seti boyutu (test_df): {test_df.shape}")

    # Dengesizlik ağırlığını (k) hesapla
    k = (y == 0).sum() / (y == 1).sum()
    print(f"Hesaplanan K (ağırlık) değeri: {k:.4f}")

except FileNotFoundError as e:
    print(f"HATA: Dosya bulunamadı. Lütfen dosya yollarını kontrol edin. {e}")
    # Kodun devam etmemesi için burada durdurabiliriz veya demo verisi oluşturabiliriz.
    # Bu senaryoda durduruyoruz.
    raise e

# =============================================================================
# ADIM 2: MODEL PARAMETRELERİNİ TANIMLA (METRİK GÜNCELLENDİ)
# =============================================================================

# Model A: Genelleyici parametreler
params_A = {
    'objective': 'binary', 
    'metric': 'auc', # <--- EN ÖNEMLİ DEĞİŞİKLİK: logloss'tan AUC'ye geçiş
    'boosting_type': 'gbdt',
    'n_estimators': 2500, 'learning_rate': 0.01, 'num_leaves': 20, 'max_depth': 5,
    'seed': 42, 'n_jobs': -1, 'verbose': -1, 'colsample_bytree': 0.7,
    'subsample': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 
    'scale_pos_weight': k
}

# Model B: Optimize edilmiş "uzman" parametreler
params_B = {
    'objective': 'binary', 
    'metric': 'auc', # <--- EN ÖNEMLİ DEĞİŞİKLİK: logloss'tan AUC'ye geçiş
    'boosting_type': 'gbdt',
    'n_estimators': 2500, 'seed': 1337, 'n_jobs': -1, 'verbose': -1, 'scale_pos_weight': k,
    'learning_rate': 0.014373, 'num_leaves': 34, 'max_depth': 3, 'subsample': 0.877,
    'colsample_bytree': 0.637, 'reg_alpha': 0.0289, 'reg_lambda': 0.3654
}

# =============================================================================
# ADIM 3: GÜÇLENDİRİLMİŞ EĞİTİM VE TAHMİN FONKSİYONU
# =============================================================================

N_SPLITS = 10 # <--- İsteğiniz üzerine 5'ten 10'a çıkarıldı

def train_and_predict(params, model_name, random_seed, n_splits):
    print(f"\n
### {model_name} Modeli Eğitiliyor... ({n_splits}-katlı CV) ###")
    
    # OOF (Out-of-Fold) tahminleri ve test tahminleri için diziler
    oof_preds = np.zeros(len(X)) 
    test_preds = np.zeros(len(test_df))
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
    params['seed'] = random_seed # Modellerin farklı olması için seed'i ayarla
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"===== Fold {fold+1}/{n_splits} =====")
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, 
                  eval_set=[(X_val, y_val)],
                  eval_metric='auc', # <--- Early stopping için de AUC kullan
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        
        # OOF tahminlerini kaydet
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        
        # Test tahminlerini topla (sonra ortalaması alınacak)
        test_preds += model.predict_proba(test_df)[:, 1] / n_splits
    
    # Bu model için OOF ve test tahminlerini döndür
    return oof_preds, test_preds

# Modelleri eğit ve tahminleri al
oof_preds_A, test_preds_A = train_and_predict(params_A, "Model A (Generalist)", 42, N_SPLITS)
oof_preds_B, test_preds_B = train_and_predict(params_B, "Model B (Uzman)", 1337, N_SPLITS)

# =============================================================================
# ADIM 4: LOKAL OOF SKORLARINI DEĞERLENDİRME (ÇOK ÖNEMLİ)
# =============================================================================
# Artık hangi modelin ve hangi karışımın en iyi Gini/AUC'yi verdiğini görebiliriz.

print("\n
### LOKAL (OOF) SKOR DEĞERLENDİRMESİ ###")

def calculate_gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    return auc, gini

auc_A, gini_A = calculate_gini(y, oof_preds_A)
auc_B, gini_B = calculate_gini(y, oof_preds_B)

print(f"Model A OOF AUC: {auc_A:.6f} | Gini: {gini_A:.6f}")
print(f"Model B OOF AUC: {auc_B:.6f} | Gini: {gini_B:.6f}")

# İki modelin tahminlerini 50/50 ağırlıkla birleştir (Blend)
blended_oof = (oof_preds_A * 0.5) + (oof_preds_B * 0.5)
blended_test_preds = (test_preds_A * 0.5) + (test_preds_B * 0.5)

auc_blend, gini_blend = calculate_gini(y, blended_oof)
print(f"Blend (50/50) OOF AUC: {auc_blend:.6f} | Gini: {gini_blend:.6f}")

# =============================================================================
# ADIM 5: FİNAL GÖNDERİM DOSYASINI OLUŞTURMA
# =============================================================================
print("\n
Final gönderim dosyası (blended) oluşturuluyor...")

submission_df = pd.DataFrame({'cust_id': sample_submission_df['cust_id'], 'churn': blended_test_preds})
submission_df.to_csv('submission_blended_final.csv', index=False)

print("Tahminleri içeren 'submission_blended_final.csv' dosyası oluşturuldu.")
print("\n
### TÜM İŞLEMLER BAŞARIYLA TAMAMLANDI! ###")
display(submission_df.head())\

In [ ]:
# === ACİL EYLEM KODU: SADECE MODEL B'Yİ GÖNDER ===

# Bu değişkenlerin bir önceki kodunuzdan geldiğini varsayıyoruz:
# test_preds_B: Model B'nin 10-fold CV ile ürettiği test seti tahminleri
# sample_submission_df: Örnek gönderim dosyası (cust_id'leri almak için)

print("\n
### Sadece Model B için Gönderim Dosyası Oluşturuluyor... ###")

# Model B'nin tahminlerini kullanarak submission_df'i oluştur
submission_model_B_only_df = pd.DataFrame({
    'cust_id': sample_submission_df['cust_id'], 
    'churn': test_preds_B  # Sadece Model B'nin tahminlerini kullan
})

# Yeni dosyayı kaydet
submission_model_B_only_df.to_csv('submission_model_B_only.csv', index=False)

print("Tahminleri içeren 'submission_model_B_only.csv' dosyası oluşturuldu.")
print("Lütfen yarışmaya BU DOSYAYI gönderin.")
display(submission_model_B_only_df.head())\

In [ ]:
# === DAHA İYİ STRATEJİ: BLEND AĞIRLIKLARINI OPTİMİZE ETME ===

# Bu değişkenlerin bir önceki kodunuzdan geldiğini varsayıyoruz:
# y: Gerçek etiketler
# oof_preds_A: Model A'nın OOF tahminleri
# oof_preds_B: Model B'nin OOF tahminleri

from sklearn.metrics import roc_auc_score

def calculate_gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    return gini

best_gini = 0
best_weight_A = 0

print("\n
### En İyi Blend Ağırlığı Aranıyor... ###")

# 0.0'dan 1.0'a kadar 0.05'lik adımlarla en iyi ağırlığı ara
for w_A in np.arange(0, 1.05, 0.05):
    w_B = 1.0 - w_A
    
    # OOF tahminlerini bu ağırlıklarla karıştır
    blended_oof = (oof_preds_A * w_A) + (oof_preds_B * w_B)
    
    # Karışımın Gini skorunu hesapla
    gini = calculate_gini(y, blended_oof)
    
    print(f"Ağırlık (A={w_A:.2f}, B={w_B:.2f}) -> OOF Gini: {gini:.6f}")
    
    if gini > best_gini:
        best_gini = gini
        best_weight_A = w_A

best_weight_B = 1.0 - best_weight_A

print("\n
--- SONUÇ ---")
print(f"En İyi OOF Gini: {best_gini:.6f}")
print(f"En İyi Ağırlık (Model A için): {best_weight_A:.2f}")
print(f"En İyi Ağırlık (Model B için): {best_weight_B:.2f}")


# === BU AĞIRLIKLARLA YENİ BİR GÖNDERİM DOSYASI OLUŞTUR ===

print("\n
En iyi ağırlıklarla final gönderim dosyası oluşturuluyor...")

# Test tahminlerini en iyi ağırlıklarla birleştir
final_blended_preds = (test_preds_A * best_weight_A) + (test_preds_B * best_weight_B)

submission_optimal_blend_df = pd.DataFrame({
    'cust_id': sample_submission_df['cust_id'], 
    'churn': final_blended_preds
})

submission_optimal_blend_df.to_csv('submission_optimal_blend.csv', index=False)
print("Tahminleri içeren 'submission_optimal_blend.csv' dosyası oluşturuldu.")\

In [ ]:
!pip install optuna catboost\

In [ ]:
import optuna
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# Önceki koddan X, y ve k (ağırlık) değişkenlerinin geldiğini varsayıyoruz.

def get_cv_auc(model, X, y, fit_params=None):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_auc = []
    for train_idx, val_idx in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        if fit_params:
            # For LightGBM and CatBoost early stopping with eval_set
            fit_kwargs = fit_params.copy()
            if 'eval_set' not in fit_kwargs:
                fit_kwargs['eval_set'] = [(X_val, y_val)]
            model.fit(X_train, y_train, **fit_kwargs)
        else:
            model.fit(X_train, y_train)
            
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_auc.append(roc_auc_score(y_val, val_preds))
    return np.mean(oof_auc)

print("### ADIM 1: LightGBM (LGBM) Optimizasyonu Başlatılıyor... ###")

def objective_lgbm(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_estimators': 2500,
        'seed': 42,
        'n_jobs': -1,
        'verbose': -1,
        'scale_pos_weight': k,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),
        'num_leaves': trial.suggest_int('num_leaves', 20, 60),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
    }

    model = lgb.LGBMClassifier(**params)
    fit_params = {'eval_metric': 'auc', 'callbacks': [lgb.early_stopping(100, verbose=False)]}
    return get_cv_auc(model, X, y, fit_params)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=5)

print("\n
--- LGBM Optimizasyonu Tamamlandı ---")
print(f"En İyi Ortalama AUC: {study_lgbm.best_value:.6f}")
print("En İyi Parametreler (LGBM):")
print(study_lgbm.best_params)
params_LGBM_optimized = {**params_A, **study_lgbm.best_params} if 'params_A' in globals() else study_lgbm.best_params


print("\n
### ADIM 2: CatBoost Optimizasyonu Başlatılıyor... ###")

def objective_catboost(trial):
    params = {
        'iterations': 2500,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': False,
        'scale_pos_weight': k,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }

    model = CatBoostClassifier(**params)
    fit_params = {'early_stopping_rounds': 100}
    return get_cv_auc(model, X, y, fit_params)

study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(objective_catboost, n_trials=5)

print("\n
--- CatBoost Optimizasyonu Tamamlandı ---")
print(f"En İyi Ortalama AUC: {study_catboost.best_value:.6f}")
print("En İyi Parametreler (CatBoost):")
print(study_catboost.best_params)
params_CatBoost_optimized = study_catboost.best_params
\

In [ ]:
from sklearn.linear_model import LogisticRegression

# Önceki kodlardan gelen değişkenleri ve Optuna'dan aldığımız parametreleri kullanıyoruz
# X, y, test_df, k
# sample_submission_df (gönderim dosyası için)

# ÖNEMLİ: Optuna'nın verdiği en iyi parametreleri buraya kopyalayın
params_LGBM_optimized = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'n_estimators': 2500, 'seed': 42, 'n_jobs': -1, 'verbose': -1, 'scale_pos_weight': k,
    **study_lgbm.best_params 
}

params_CatBoost_optimized = {
    'iterations': 2500, 'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'random_seed': 42, 'verbose': False, 'scale_pos_weight': k,
    'early_stopping_rounds': 100,
    **study_catboost.best_params 
}

print("\n
### ADIM 3: Optimize Edilmiş Modellerle Final Eğitimi Başlıyor... ###")

def train_and_evaluate(model, X, y, test_df, model_name, n_splits=10):
    print(f"\n
--- Optimize {model_name} Modeli Eğitiliyor ---")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(test_df))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"{model_name} Fold {fold+1}/{n_splits}")
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        if model_name == 'LGBM':
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(100, verbose=False)])
        else:
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
            
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(test_df)[:, 1] / n_splits
        
    return oof_preds, test_preds

oof_preds_lgbm, test_preds_lgbm = train_and_evaluate(lgb.LGBMClassifier(**params_LGBM_optimized), X, y, test_df, 'LGBM')
oof_preds_catboost, test_preds_catboost = train_and_evaluate(CatBoostClassifier(**params_CatBoost_optimized), X, y, test_df, 'CatBoost')

print("\n
--- Final Modelleri Eğitildi! ---")

# === Lokal (OOF) Skorları Değerlendirme ===
def calculate_gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    return auc, gini

auc_lgbm, gini_lgbm = calculate_gini(y, oof_preds_lgbm)
auc_catboost, gini_catboost = calculate_gini(y, oof_preds_catboost)

print(f"Optimize LGBM OOF AUC: {auc_lgbm:.6f} | Gini: {gini_lgbm:.6f}")
print(f"Optimize CatBoost OOF AUC: {auc_catboost:.6f} | Gini: {gini_catboost:.6f}")

# =============================================================================
# ADIM 4: STACKING (NİHAİ BİRLEŞTİRME)
# =============================================================================
print("\n
### ADIM 4: Stacking ile Modeller Birleştiriliyor... ###")

X_stack_train = pd.DataFrame({
    'lgbm_pred': oof_preds_lgbm,
    'catboost_pred': oof_preds_catboost
})

X_stack_test = pd.DataFrame({
    'lgbm_pred': test_preds_lgbm,
    'catboost_pred': test_preds_catboost
})

meta_model = LogisticRegression()
meta_model.fit(X_stack_train, y)

oof_preds_stacking = meta_model.predict_proba(X_stack_train)[:, 1]
auc_stack, gini_stack = calculate_gini(y, oof_preds_stacking)

print(f"\n
Stacking Meta-Model OOF AUC: {auc_stack:.6f} | Gini: {gini_stack:.6f}")
final_test_preds = meta_model.predict_proba(X_stack_test)[:, 1]

# =============================================================================
# FİNAL GÖNDERİM DOSYASI
# =============================================================================
print("\n
Final gönderim dosyası (Stacking) oluşturuluyor...")

submission_df = pd.DataFrame({
    'cust_id': sample_submission_df['cust_id'], 
    'churn': final_test_preds
})
submission_df.to_csv('submission_stacked_final.csv', index=False)

print("Tahminleri içeren 'submission_stacked_final.csv' dosyası oluşturuldu.")
print("\n
### TÜM İŞLEMLER BAŞARIYLA TAMAMLANDI! ###")
display(submission_df.head())
\